# DRI-18 Run #3 Colab Diagnostic

CUDA/Colab workflow for a diagnostic-first QLoRA run. This notebook regenerates TBX11K JSONL locally, checks the runtime, trains in checkpointed segments, evaluates early class-collapse gates, and uploads artifacts to Hugging Face.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/ShivamSinghNow/Drishti.git"
BRANCH = "codex/dri-18-colab-run3"  # switch to main after this branch is merged
WORKDIR = Path("/content/Drishti")

RUN_NAME = "drishti-qlora-run3-colab-soft-balanced-diagnostic"
OUTPUT_DIR = Path("outputs/dri18-run3-colab-soft-balanced-diagnostic")
EVAL_ROOT = Path("outputs/eval/dri18-run3-colab-soft-balanced-diagnostic")
HF_REPO_ID = "ShivSingh123/drishti-qlora-run3-colab-soft-balanced-diagnostic"

DIAGNOSTIC_STEPS = [200, 400, 800]
VAL_EVAL_LIMIT = 450
RUN_FULL_AFTER_DIAGNOSTIC = False

In [ ]:
if not WORKDIR.exists():
    !git clone --branch {BRANCH} {REPO_URL} {WORKDIR}
else:
    %cd {WORKDIR}
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}

%cd {WORKDIR}
!python -m pip install -q --upgrade pip setuptools wheel
!python -m pip install -q -r requirements-colab.txt

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import HfApi, login

def require_secret(name: str) -> str:
    value = userdata.get(name)
    if not value:
        raise RuntimeError(f"Missing Colab secret: {name}")
    return value

os.environ["KAGGLE_USERNAME"] = require_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = require_secret("KAGGLE_KEY")
os.environ["WANDB_API_KEY"] = require_secret("WANDB_API_KEY")
os.environ["HF_TOKEN"] = require_secret("HF_TOKEN")

login(token=os.environ["HF_TOKEN"])
api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(repo_id=HF_REPO_ID, repo_type="model", private=True, exist_ok=True)

In [ ]:
import importlib.metadata as metadata
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > GPU before continuing.")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"gpu={gpu_name} vram_gb={gpu_mem_gb:.1f}")
print(f"torch={torch.__version__} cuda={torch.version.cuda}")
for package in ("transformers", "peft", "bitsandbytes", "accelerate", "wandb"):
    print(f"{package}={metadata.version(package)}")

preferred = any(name in gpu_name for name in ("A100", "L4"))
t4_smoke_only = "T4" in gpu_name
if not preferred:
    if t4_smoke_only:
        raise RuntimeError("T4 detected. Use this notebook for smoke/setup only; reconnect until Colab gives A100 or L4 for diagnostic training.")
    raise RuntimeError(f"Unexpected GPU for run #3: {gpu_name}. Reconnect for A100 or L4.")

In [ ]:
!python download_dataset.py
!python generate_jsonl.py --output-dir data/processed

import json
from collections import Counter

expected = {
    "train": Counter({"active_tb": 600, "healthy": 3000, "sick_but_non_tb": 3000}),
    "val": Counter({"active_tb": 200, "healthy": 800, "sick_but_non_tb": 800}),
}

for split, expected_counts in expected.items():
    counts = Counter()
    with open(f"data/processed/{split}.jsonl", encoding="utf-8") as handle:
        for line in handle:
            payload = json.loads(line)
            assistant = payload["messages"][1]["content"]
            assert assistant.startswith("Classification: "), assistant
            assert assistant.count("\n") == 0, assistant
            counts[assistant.removeprefix("Classification: ")] += 1
    print(split, dict(sorted(counts.items())))
    assert counts == expected_counts, (split, counts, expected_counts)

In [ ]:
!python -m unittest -v
!python train_qlora.py --dry-run --train-limit 2 --eval-limit 2
!python train_qlora.py --setup-only --train-limit 2 --eval-limit 2 --wandb-mode disabled --output-dir outputs/dri18-setup-validation

In [ ]:
import subprocess

os.environ["WANDB_PROJECT"] = "tbx11k-qwen-vl-finetuning"
os.environ["WANDB_MODE"] = "online"
os.environ["WANDB_TAGS"] = "dri-18,run3,colab,diagnostic,soft-balanced"
os.environ["WANDB_NOTES"] = "Run #3 diagnostic: CUDA Colab, softened balanced sampler, no sick boost, segmented early gates."
os.environ["WANDB_RUN_ID"] = "dri18-run3-colab-soft-balanced-diagnostic"
os.environ["WANDB_RESUME"] = "allow"

def run(command: list[str]) -> None:
    print("\n$ " + " ".join(command))
    subprocess.run(command, check=True)

def upload_path(local_path: Path, repo_path: str) -> None:
    if not local_path.exists():
        raise FileNotFoundError(local_path)
    api.upload_folder(
        repo_id=HF_REPO_ID,
        repo_type="model",
        folder_path=str(local_path),
        path_in_repo=repo_path,
    )

base_train_cmd = [
    "python", "train_qlora.py",
    "--run-name", RUN_NAME,
    "--output-dir", str(OUTPUT_DIR),
    "--rank", "32",
    "--alpha", "64",
    "--lora-dropout", "0.05",
    "--lr", "1.5e-4",
    "--batch-size", "1",
    "--grad-accum", "4",
    "--warmup-steps", "150",
    "--weight-decay", "0.01",
    "--lr-scheduler-type", "cosine",
    "--sampling-strategy", "soft-balanced",
    "--save-steps", "200",
    "--eval-steps", "200",
    "--logging-steps", "10",
    "--seed", "42",
]

previous_checkpoint = None
for target_step in DIAGNOSTIC_STEPS:
    train_cmd = base_train_cmd + ["--max-steps", str(target_step)]
    if previous_checkpoint is not None:
        train_cmd += ["--resume-from-checkpoint", str(previous_checkpoint)]
    run(train_cmd)

    checkpoint = OUTPUT_DIR / f"checkpoint-{target_step}"
    eval_dir = EVAL_ROOT / f"checkpoint-{target_step}"
    try:
        run([
            "python", "evaluate_checkpoint.py",
            "--adapter-dir", str(checkpoint),
            "--data-dir", "data/processed",
            "--split", "val",
            "--output-dir", str(eval_dir),
            "--batch-size", "3",
            "--limit", str(VAL_EVAL_LIMIT),
            "--gate", "run3-diagnostic",
            "--fail-on-gate-fail",
        ])
    finally:
        upload_path(checkpoint, f"checkpoints/checkpoint-{target_step}")
        if eval_dir.exists():
            upload_path(eval_dir, f"eval/checkpoint-{target_step}")

    previous_checkpoint = checkpoint

print("Diagnostic segments passed. Full continuation remains opt-in in the next cell.")

In [ ]:
if RUN_FULL_AFTER_DIAGNOSTIC:
    if previous_checkpoint is None:
        raise RuntimeError("Run the diagnostic cell first.")
    run(base_train_cmd + [
        "--epochs", "3",
        "--max-steps", "-1",
        "--resume-from-checkpoint", str(previous_checkpoint),
    ])
    final_eval_dir = EVAL_ROOT / "final-full-val"
    run([
        "python", "evaluate_checkpoint.py",
        "--adapter-dir", str(OUTPUT_DIR),
        "--data-dir", "data/processed",
        "--split", "val",
        "--output-dir", str(final_eval_dir),
        "--batch-size", "3",
        "--gate", "run3-full",
        "--fail-on-gate-fail",
    ])
    upload_path(OUTPUT_DIR, "final-adapter-and-checkpoints")
    upload_path(final_eval_dir, "eval/final-full-val")
else:
    print("RUN_FULL_AFTER_DIAGNOSTIC is False. Review W&B and checkpoint evals before continuing.")